In [0]:
%run ../logging_utils

In [0]:
import json
import ast
from datetime import datetime
from dateutil.relativedelta import relativedelta

In [0]:
RunId = dbutils.widgets.get("RunId")
cod_pipelineID = dbutils.widgets.get("cod_pipelineID")
cod_layerName = dbutils.widgets.get("cod_layerName")
snapshot_date = dbutils.widgets.get("snapshot_date")
result_state = dbutils.widgets.get("result_state")
error_code = dbutils.widgets.get("error_code")
input_params = dbutils.widgets.get("input_params")
env = dbutils.widgets.get("env")
cod_productName = dbutils.widgets.get("cod_productName")

In [0]:
general_utils = LoggingUtils(spark)
logger = general_utils.set_logging()


try:
    input_params_json = ast.literal_eval(input_params)
except (ValueError, SyntaxError) as e:
    input_params_json = json.loads(input_params)


snapshot_date = general_utils.date_to_start_month(snapshot_date)
try:
    snapshot_date = general_utils.replace_month_date(snapshot_date, int(input_params_json['today_minus_n_months']))
except:
    snapshot_date = general_utils.replace_month_date(snapshot_date, 0)
logger.info(f"Snapshot_date materialized: {snapshot_date}")

try:
    logger.info(f"METRICS Log generation process INITIALIZING")
    config = general_utils.load_config(general_utils.get_variable(cod_productName)['config_setup_path'])
    snapshot_date = general_utils.date_to_start_month(snapshot_date)
    catalog_name = config['environment'][env]['catalog']
    schema_name = config['environment'][env]['schema']

    # logger.info(f"Get parameters")
    # params = general_utils.get_pipeline_parameters()

    logger.info(f"Get table reference")
    log_catalog, log_database, log_table = general_utils.get_log_processor_reference(catalog_name, 
                                "monitoring", 
                                config['logging']['processor']['table'])

    logger.info(f"Formatting table reference")
    try:
        input_params_json = ast.literal_eval(input_params)
    except (ValueError, SyntaxError) as e:
        input_params_json = json.loads(input_params)
except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

In [0]:
try:
    logger.info(f"Closing log with Error state on {log_table} table")
    
    general_utils.close_process_log(
        catalog=catalog_name, 
        database=log_database, 
        table=log_table, 
        metrica=input_params_json['name'], 
        cod_runID=RunId, 
        is_error=True, 
        error_msg=f"{result_state} - {error_code}"
    )
    
    logger.info(f"Log generation process END")
except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

In [0]:
# RunId = dbutils.widgets.get("RunId")
# cod_pipelineID = dbutils.widgets.get("cod_pipelineID")
# cod_layerName = dbutils.widgets.get("cod_layerName")
# snapshot_date = dbutils.widgets.get("snapshot_date")
# result_state = dbutils.widgets.get("result_state")
# error_code = dbutils.widgets.get("error_code")
# input_params = dbutils.widgets.get("input_params")
# env = dbutils.widgets.get("env")

# general_utils = LoggingUtils(spark)
# config = general_utils.load_config(general_utils.get_variable()['config_setup_path'])

# params = general_utils.get_pipeline_parameters()

# snapshot_date = general_utils.date_to_start_month(snapshot_date)
# catalog_name = config['environment'][env]['catalog']
# schema_name = config['environment'][env]['schema']

# try:
#     input_params_json = ast.literal_eval(input_params)
# except (ValueError, SyntaxError) as e:
#     input_params_json = json.loads(input_params)

# try:
#     snapshot_date = general_utils.replace_month_date(snapshot_date, int(input_params_json['today_minus_n_months']))
# except:
#     snapshot_date = general_utils.replace_month_date(snapshot_date, 0)
# logger.info(f"Snapshot_date materialized: {snapshot_date}")



# log_catalog, log_database, log_table = general_utils.get_log_processor_reference(catalog_name, 
#                             "monitoring", 
#                             config['logging']['processor']['table'])

# try:
#     input_params_json = ast.literal_eval(input_params)
# except (ValueError, SyntaxError) as e:
#     input_params_json = json.loads(input_params)

In [0]:
# general_utils.populate_monitor_log_date(catalog_name, log_database, log_table, input_params_json['name'], RunId, is_started=False)

# general_utils.populate_log_error(catalog_name, log_database, log_table, input_params_json['name'], params['snapshot_date'], dbutils.widgets.get("error_code"), RunId, is_error=True)

# general_utils.populate_log_state(catalog_name, log_database, log_table, input_params_json['name'], params['snapshot_date'], RunId, is_error=True)

In [0]:
print("Application parameters", input_params_json)

In [0]:
dbutils.notebook.exit("Success!!")

In [0]:
%sql
select * from p_sbx_data.data.ih_metrica_lg_processmonitor
where metric_name = 'MetricaFlgVoltureLordeNette'
order by dat_processStartDate desc